# Playbook · Smart money, point in time

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/valuein/valuein/blob/main/examples/playbooks/smart-money-point-in-time.ipynb)

**Who it is for:** portfolio managers and analysts who track insiders, 13F holders and blockholders.

**What you get:** insider trades, 13F holders and a manager's portfolio, each with the time the SEC accepted the filing, and the rule that keeps a back-test from reading a 13F before it existed.

**Plan needed:** Institutional for the data cells. Without an Institutional key every data cell prints a one-line skip (the plan error) and the notebook still runs to the end. The schema step runs on any plan.

## Step 1 · Open the client and check your plan

Smart-money tables (insider Forms 3/4/5, 13F, 13D/13G, Form ADV) are Institutional-only. The client tells you which plan it is on before any request is spent.

In [1]:
%pip install -q "valuein-sdk>=1.1.0" matplotlib

Note: you may need to restart the kernel to use updated packages.


In [2]:
import warnings

import pandas as pd

from valuein_sdk import ValueinClient, ValueinError

warnings.filterwarnings("ignore", message="Defaulting to SAMPLE dataset")
client = ValueinClient()
print(f"plan: {client.plan!r}")


def skip(exc: Exception) -> None:
    """Print a one-line skip with the plan error; the notebook keeps going."""
    print(f"Skipped: {exc}")
    if client.plan != "full":
        print("To run this cell set VALUEIN_API_KEY to an Institutional key: https://valuein.biz/pricing")

plan: 'full'


### See what the tables hold

Column lists come from the manifest, so this works on every plan and downloads nothing.

In [3]:
for table in ["insider_transaction", "institutional_holding", "investment_adviser"]:
    columns = list(client.get_schema(table))
    print(f"{table:<22} {len(columns):>3} columns: {', '.join(columns[:8])}, ...")

insider_transaction     39 columns: id, accession_id, transaction_line_no, entity_id, insider_party_id, form_type, transaction_type, transaction_date, ...
institutional_holding   25 columns: id, accession_id, filer_cik, filer_name, period_end, share_class_figi, subject_entity_id, name_of_issuer, ...
investment_adviser      61 columns: id, crd_number, filing_date, sec_number, business_name, legal_name, sec_region, umbrella_registration, ...


**Check it worked:** it prints your plan (`'sample'` with no key, `'full'` on Institutional).

Then: three tables, each with its column count and the first columns.

## Step 2 · Insider buys and sales

Open-market purchases (`P`) and sales (`S`) from Forms 4. `accepted_at` is when the SEC accepted the filing: the moment the trade became public.

In [4]:
insiders = None
try:
    insiders = client.insider_transactions("NVDA", transaction_codes=["P", "S"], limit=50)
    print(insiders[["transaction_date", "insider_name", "insider_role", "transaction_code",
                    "shares", "accepted_at"]].head(10).to_string(index=False))
except ValueinError as exc:
    skip(exc)

transaction_date   insider_name insider_role transaction_code    shares               accepted_at
      2026-09-21      Teter Tim    [Officer]                S   12483.0 2026-09-24 02:00:00+00:00
      2026-09-21      Teter Tim    [Officer]                S   13478.0 2026-09-24 02:00:00+00:00
      2026-09-21      Teter Tim    [Officer]                S    4499.0 2026-09-24 02:00:00+00:00
      2026-09-18 STEVENS MARK A   [Director]                S 1356000.0 2026-09-23 02:00:00+00:00
      2026-09-18 STEVENS MARK A   [Director]                S   10000.0 2026-09-23 02:00:00+00:00
      2026-09-17  Kress Colette    [Officer]                S    4681.0 2026-09-19 02:00:00+00:00
      2026-09-17  Kress Colette    [Officer]                S   20369.0 2026-09-19 02:00:00+00:00
      2026-09-17  Kress Colette    [Officer]                S    9868.0 2026-09-19 02:00:00+00:00
      2026-09-04 STEVENS MARK A   [Director]                S  197180.0 2026-09-09 02:00:00+00:00
      2026-09-04 STE

### Who holds a company: 13F holders

13F holders of one issuer, latest reported quarter. Each row has a `period_end` (the quarter) and an `accepted_at` (when the filing became public, up to 45 days later).

In [5]:
holders = None
try:
    holders = client.institutional_holdings(issuer="NVDA", limit=200)
    latest = holders[holders["period_end"] == holders["period_end"].max()]
    print(latest[["filer_name", "shares", "market_value_usd", "period_end", "accepted_at"]]
          .sort_values("market_value_usd", ascending=False).head(10).to_string(index=False))
except ValueinError as exc:
    skip(exc)

                       filer_name       shares  market_value_usd period_end               accepted_at
  VANGUARD CAPITAL MANAGEMENT LLC 1540600742.0      3.518424e+11 2026-09-30 2026-11-13 16:56:18+00:00
                STATE STREET CORP 1025075347.0      2.343556e+11 2026-09-30 2026-11-12 11:29:20+00:00
                  BlackRock, Inc.  738850605.0      1.687387e+11 2026-09-30 2026-11-13 20:52:08+00:00
                          FMR LLC  638423321.0      1.458031e+11 2026-09-30 2026-11-10 14:26:26+00:00
    GEODE CAPITAL MANAGEMENT, LLC  535882793.0      1.220162e+11 2026-09-30 2026-11-06 18:10:13+00:00
VANGUARD PORTFOLIO MANAGEMENT LLC  512235499.0      1.169843e+11 2026-09-30 2026-11-13 16:56:14+00:00
                  BlackRock, Inc.  489185028.0      1.117201e+11 2026-09-30 2026-11-13 20:52:08+00:00
              JPMORGAN CHASE & CO  275240891.0      6.253748e+10 2026-09-30 2026-11-12 13:25:09+00:00
                     Invesco Ltd.  242846423.0      5.546127e+10 2026-09-30 2026-1

**Check it worked:** ten rows with a transaction date and an `accepted_at` after it, or a one-line `Skipped:` message.

Then: the largest holders of the latest quarter with `period_end` and `accepted_at`, or a one-line `Skipped:` message.

## Step 3 · A manager's portfolio

Turn the question around: one 13F filer's holdings. The weight below is computed here from the reported market values, not stored by Valuein.

In [6]:
portfolio = None
try:
    portfolio = client.institutional_holdings(filer_cik="0001067983", limit=2000)  # Berkshire Hathaway
    latest = portfolio[portfolio["period_end"] == portfolio["period_end"].max()].copy()
    latest["weight_pct (computed here)"] = 100 * latest["market_value_usd"] / latest["market_value_usd"].sum()
    print(latest[["name_of_issuer", "shares", "market_value_usd", "weight_pct (computed here)"]]
          .sort_values("market_value_usd", ascending=False).head(10).round(2).to_string(index=False))
except ValueinError as exc:
    skip(exc)

      name_of_issuer      shares  market_value_usd  weight_pct (computed here)
 AMERICAN EXPRESS CO 149061045.0      5.041990e+10                       16.85
        ALPHABET INC  65824467.0      2.352369e+10                        7.86
           APPLE INC  80664820.0      2.334117e+10                        7.80
        COCA COLA CO 282722729.0      2.297688e+10                        7.68
           APPLE INC  61542988.0      1.780808e+10                        5.95
   BANK OF AMER CORP 308808515.0      1.759591e+10                        5.88
OCCIDENTAL PETE CORP 264941431.0      1.286821e+10                        4.30
       CHUBB LIMITED  34249183.0      1.167007e+10                        3.90
           APPLE INC  34724000.0      1.004774e+10                        3.36
 CHEVRON CORPORATION  58251749.0      9.655810e+09                        3.23


**Check it worked:** the filer's largest positions for its latest quarter, or a one-line `Skipped:` message.

## Step 4 · Keep it point in time

A 13F for the quarter ending 30 June is filed by mid-August. Gate on `accepted_at`, never on `period_end`, or your backtest holds a position weeks before anyone could have known about it.

In [7]:
if portfolio is None:
    print("Skipped: needs the Institutional plan (see step 3).")
else:
    quarters = sorted(portfolio["period_end"].dropna().unique())
    if len(quarters) < 3:
        print("Need three quarters of filings here: raise limit= in step 3.")
    else:
        # A cutoff 30 days after a quarter ends: the quarter is over, but its 13F
        # (due 45 days after quarter end) has usually not been filed yet.
        quarter = pd.Timestamp(quarters[-2])
        CUTOFF = (quarter + pd.Timedelta(days=30)).tz_localize("UTC")
        wrong = portfolio[portfolio["period_end"] <= CUTOFF.tz_localize(None)]
        right = portfolio[portfolio["accepted_at"] <= CUTOFF]
        print(f"cutoff: {CUTOFF:%Y-%m-%d}")
        print(f"latest quarter by period_end  <= cutoff: {wrong['period_end'].max():%Y-%m-%d}  (look-ahead)")
        if right.empty:
            print("latest quarter by accepted_at <= cutoff: none in the rows fetched; raise limit= in step 3")
        else:
            print(f"latest quarter by accepted_at <= cutoff: {right['period_end'].max():%Y-%m-%d}  (point in time)")

cutoff: 2026-04-30
latest quarter by period_end  <= cutoff: 2026-03-31  (look-ahead)
latest quarter by accepted_at <= cutoff: 2025-12-31  (point in time)


In [8]:
if portfolio is None:
    print("Skipped: needs the Institutional plan (see step 3).")
else:
    accession = portfolio.sort_values("accepted_at").iloc[-1].get("accession_id")
    if accession:
        folder = accession.replace("-", "")
        print("Open the filing on sec.gov:", f"https://www.sec.gov/Archives/edgar/data/1067983/{folder}/")
    client.close()

Open the filing on sec.gov: https://www.sec.gov/Archives/edgar/data/1067983/000119312526352200/


**Check it worked:** the two rules name different quarters when the cutoff falls between the quarter end and the filing, and the last cell prints a sec.gov link (or a skip).

**Next:** [Backtest without bias](backtest-without-bias.ipynb) · [Smart money notebook](../notebooks/07_smart_money.ipynb)

_Index names are trademarks of their owners; Valuein is not affiliated with or endorsed by any index provider._